## What is an Agent?

An autonomous entity that:
- **Perceives** its environment
- **Decides** on actions
- **Acts** to achieve goals
- **Learns** from feedback

### Agent Architecture:
```
Environment Observation
        ↓
Perception/Sensing
        ↓
Decision Making (Policy)
        ↓
Action Selection
        ↓
Environment Update
        ↓
Reward/Feedback
```

In [ ]:
import numpy as np
import random
from collections import defaultdict

# Simple GridWorld Environment
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.agent_pos = (0, 0)
        self.goal_pos = (size-1, size-1)
    
    def reset(self):
        self.agent_pos = (0, 0)
        return self.get_state()
    
    def get_state(self):
        return self.agent_pos
    
    def step(self, action):
        """Execute action: 0=up, 1=down, 2=left, 3=right"""
        x, y = self.agent_pos
        
        if action == 0:  # up
            x = max(0, x - 1)
        elif action == 1:  # down
            x = min(self.size - 1, x + 1)
        elif action == 2:  # left
            y = max(0, y - 1)
        elif action == 3:  # right
            y = min(self.size - 1, y + 1)
        
        self.agent_pos = (x, y)
        
        # Reward
        if self.agent_pos == self.goal_pos:
            reward = 1
            done = True
        else:
            reward = -0.01  # Small penalty for each step
            done = False
        
        return self.get_state(), reward, done

# Simple Q-Learning Agent
class QLearningAgent:
    def __init__(self, grid_size=5, learning_rate=0.1, discount_factor=0.9, epsilon=0.1):
        self.grid_size = grid_size
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.q_table = defaultdict(lambda: np.zeros(4))
    
    def select_action(self, state, training=True):
        """Epsilon-greedy action selection"""
        if training and random.random() < self.epsilon:
            return random.randint(0, 3)  # Random action
        else:
            return np.argmax(self.q_table[state])  # Greedy action
    
    def learn(self, state, action, reward, next_state, done):
        """Q-learning update"""
        current_q = self.q_table[state][action]
        max_next_q = np.max(self.q_table[next_state])
        
        # Q-learning formula
        new_q = current_q + self.learning_rate * (reward + self.discount_factor * max_next_q - current_q)
        self.q_table[state][action] = new_q

# Training
env = GridWorld(size=5)
agent = QLearningAgent()

episodes = 100
rewards_per_episode = []

for episode in range(episodes):
    state = env.reset()
    total_reward = 0
    steps = 0
    
    while steps < 50:  # Max steps per episode
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        agent.learn(state, action, reward, next_state, done)
        
        total_reward += reward
        state = next_state
        steps += 1
        
        if done:
            break
    
    rewards_per_episode.append(total_reward)

print(f"Training completed. Average reward (last 10 episodes): {np.mean(rewards_per_episode[-10:]):.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot learning progress
plt.figure(figsize=(10, 5))
plt.plot(rewards_per_episode, label='Episode Reward')
plt.plot(pd.Series(rewards_per_episode).rolling(window=10).mean(), label='10-Episode Moving Average')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Q-Learning Training Progress')
plt.legend()
plt.grid()
plt.show()

# Test the agent
print("\nTesting trained agent:")
state = env.reset()
path = [state]

for _ in range(20):
    action = agent.select_action(state, training=False)
    state, reward, done = env.step(action)
    path.append(state)
    
    if done:
        break

print(f"Path taken: {path}")
print(f"Steps to goal: {len(path) - 1}")

## Types of Agents

### 1. Reactive Agents
- Simple stimulus-response
- No memory or planning
- Fast but limited

### 2. Deliberative Agents
- Make plans
- Have goals
- Use reasoning

### 3. Learning Agents
- Improve from experience
- Adapt to environment
- Use reinforcement learning

### 4. Multi-Agent Systems
- Multiple agents interact
- Cooperation and competition
- Distributed decision-making

## Real-World Applications

1. **Autonomous Vehicles**: Navigation and decision-making
2. **Robotics**: Path planning and manipulation
3. **Game AI**: NPCs and strategy
4. **Conversational AI**: Dialog management
5. **Recommendation Systems**: Contextual suggestions
6. **Trading Bots**: Autonomous trading decisions
7. **Resource Management**: Optimization and scheduling

## Practice Exercises

1. **Extend GridWorld**: Add obstacles and more complex rewards
2. **Policy Iteration**: Implement value iteration or policy iteration
3. **Deep Q-Learning**: Use neural networks with Q-learning
4. **Multi-Agent**: Create agents that interact with each other
5. **Game AI**: Build an agent for a simple game (Tic-tac-toe, Connect 4)